In [8]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit

from parallel_two_qubit_gate_decomposition import *
from parallel_two_qubit_gate_decomposition import (
    cnot_gate,
    fsim_gate,
    cphase_gate,
    xy_gate,
    get_gate_unitary_qiskit,
)

from qiskit.quantum_info import random_unitary
from tqdm import tqdm


## Decomposition example with a single two-qubit gate 

In [3]:
# CNOT gate in the circuit
my_op = np.matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

In [4]:
test_circuit = QuantumCircuit(2)
test_circuit.unitary(my_op, [0, 1])

In [5]:
test_circuit.draw()

┌──────────┐
q_0: ┤0         ├
     │  Unitary │
q_1: ┤1         ├
     └──────────┘

In [6]:
# Assume the HW supports the Google SYC gate. This is an underlying fsim_gate.
# So we pass the fsim_gate function in the gate_defs,
# we pass the desired SYC parameters (theta=pi/2, phi=pi/6) in the params, we pass a label for circuit drawing,
# and we pass the calibration fidelity of this gate on qubits 0 and 1

fid_2q = {(0, 1): [0.995]}
params = [[np.pi / 2, np.pi / 6]]
gate_labels = ["SYC"]
gate_defs = [fsim_gate]

# All these inputs get passed to the gate decomposition pass.
# Ignore the [1 to 54] loop (that was from our assumption of Sycamore hardware, but I believe it no longer plays a role)
# tolerance defines the fidelity accuracy we want from the decomposition. If you set the tol=0.1 or so, you can see that
# lesser two-qubit gates are used.

pgrp = ParallelGateReplacementPass(
    gate_defs, params, gate_labels, fid_2q, [1 for _ in range(54)], tol=0.01
)
approx_cz = pgrp.run(test_circuit)

In [7]:
approx_cz.draw()

┌─────────────────────────┐ ┌──────┐┌───────────────────────────┐┌──────┐»
q_0: ┤ U3(2.849,1.4993,5.9036) ├─┤1     ├┤ U3(6.2832,4.947,0.071623) ├┤1     ├»
     ├─────────────────────────┴┐│  SYC │└┬─────────────────────────┬┘│  SYC │»
q_1: ┤ U3(3.1416,5.1304,3.2979) ├┤0     ├─┤ U3(4.7843,4.5218,4.651) ├─┤0     ├»
     └──────────────────────────┘└──────┘ └─────────────────────────┘ └──────┘»
«     ┌──────────────────────────┐
«q_0: ┤ U3(4.8195,1.2979,0.4823) ├
«     ├──────────────────────────┤
«q_1: ┤ U3(3.1416,4.3838,2.7687) ├
«     └──────────────────────────┘

In [10]:
for _ in tqdm(range(10)):
    test_circuit = QuantumCircuit(2)
    test_circuit.append(random_unitary(4), [0, 1])
    approx_cz = pgrp.run(test_circuit)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:08<00:00,  1.16it/s]


## Decomposition example with two circuit gates with two hardware gates

In [26]:
test_circuit = QuantumCircuit(5)
test_circuit.append(random_unitary(4), [0, 1])

In [27]:
fid_2q = {
    (0, 1): [
        0.94,  # CZ fidelity
        0.99,
    ],  # XY fidelity
    (0, 1): [
        0.93,  # CZ
        0.96,
    ],
}  # XY
params = [[np.pi], [np.pi]]
gate_labels = ["CZ", "XY"]
gate_defs = [cphase_gate, xy_gate]
pgrp = ParallelGateReplacementPass(
    gate_defs, params, gate_labels, fid_2q, [1 for _ in range(54)], tol=0.00000001
)
approx = pgrp.run(test_circuit)

In [28]:
approx.draw()

┌──────────────────────────┐ ┌─────┐ ┌──────────────────────────┐┌─────┐»
q_0: ─┤ U3(4.9349,2.4912,1.6623) ├─┤1    ├─┤ U3(1.2143,3.2392,4.8963) ├┤1    ├»
     ┌┴──────────────────────────┴┐│  XY │┌┴──────────────────────────┤│  XY │»
q_1: ┤ U3(5.0014,0.98272,0.83951) ├┤0    ├┤ U3(3.8437,0.98548,3.5103) ├┤0    ├»
     └────────────────────────────┘└─────┘└───────────────────────────┘└─────┘»
q_2: ─────────────────────────────────────────────────────────────────────────»
                                                                              »
q_3: ─────────────────────────────────────────────────────────────────────────»
                                                                              »
q_4: ─────────────────────────────────────────────────────────────────────────»
                                                                              »
«     ┌────────────────────────────┐┌─────┐ ┌──────────────────────────┐ 
«q_0: ┤ U3(3.2544,0.026209,6.7736) ├┤1    ├─┤ U3(1.4035,5.4849,2.6775) ├─
«     └┬──────────────────────────┬┘│  XY │┌┴──────────────────────────┴┐
«q_1: ─┤ U3(4.8843,3.3102,5.8468) ├─┤0    ├┤ U3(0.045978,0.41264,3.649) ├
«      └──────────────────────────┘ └─────┘└────────────────────────────┘
«q_2: ───────────────────────────────────────────────────────────────────
«                                                                        
«q_3: ───────────────────────────────────────────────────────────────────
«                                                                        
«q_4: ───────────────────────────────────────────────────────────────────
«

In [29]:
for _ in tqdm(range(10)):
    test_circuit = QuantumCircuit(2)
    test_circuit.append(random_unitary(4), [0, 1])
    approx_cz = pgrp.run(test_circuit)

100%|██████████| 10/10 [00:17<00:00,  1.71s/it]
